In [1]:
import pandas as pd
import networkx as nx
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import graph
import ast

Import Bills data

In [2]:
# billsPath =  os.path.join('..', 'data', 'raw', 'final_bill_data.csv')
billsPath =  os.path.join('..', 'data', 'raw', 'boston_topic_bill_data.csv')
billsDF = pd.read_csv(billsPath, engine='python')
billsDF = billsDF[billsDF['municipality'] == 'Boston']
billsDF

,bill_number,bill_url,title,sponsor,sponsor_url,summary_title,text,source_url,view_text_url,download_pdf_url,...,municipality,num_cosponsors,cosponsors,history,path,summary_text_clean,summary_title_clean,cleaned,topic,topic_probability
118,H.5097,https://malegislature.gov/Bills/192/H5097,An Act relative to certain affordable housing ...,"Michlewitz, Aaron",https://malegislature.gov/Legislators/Profile/...,An Act relative to certain affordable housing ...,SECTION 1. Construction and development activi...,https://malegislature.gov/Bills/192/H5097,https://malegislature.gov/Bills/192/H5097/Hous...,https://malegislature.gov/Bills/192/H5097.pdf,...,Boston,0,[],"[{'date': '7/26/2022', 'branch': 'House', 'act...",../downloaded_legislation/state/MA/192nd/2025/...,SECTION 1. Construction and development activi...,An Act relative to certain affordable housing ...,certain affordable housing branch library spac...,0,0.438492
272,H.5039,https://malegislature.gov/Bills/192/H5039,An Act relative to Chapter 30B: Uniform Procur...,"Holmes, Russell E.",https://malegislature.gov/Legislators/Profile/...,An Act relative to Chapter 30B: Uniform Procur...,"In the City of Boston, notwithstanding section...",https://malegislature.gov/Bills/192/H5039,https://malegislature.gov/Bills/192/H5039/Hous...,https://malegislature.gov/Bills/192/H5039.pdf,...,Boston,0,[],"[{'date': '7/18/2022', 'branch': 'House', 'act...",../downloaded_legislation/state/MA/192nd/2025/...,NaN,An Act relative to Chapter 30B: Uniform Procur...,uniform procurement,3,1.000000
525,H.3753,https://malegislature.gov/Bills/192/H3753,An Act regarding preference for Boston high sc...,"Fluker Oakley, Brandy",https://malegislature.gov/Legislators/Profile/...,An Act regarding preference for Boston high sc...,"SECTION 1. In this act, all terms shall have t...",https://malegislature.gov/Bills/192/H3753,https://malegislature.gov/Bills/192/H3753/Hous...,https://malegislature.gov/Bills/192/H3753.pdf,...,Boston,0,[],"[{'date': '5/10/2021', 'branch': 'House', 'act...",../downloaded_legislation/state/MA/192nd/2025/...,"SECTION 1. In this act, all terms shall have t...",An Act regarding preference for Boston high sc...,preference boston high school graduate positio...,-1,0.000000
969,H.5330,https://malegislature.gov/Bills/192/H5330,An Act regarding the disability pension for El...,"Hunt, Daniel J.",https://malegislature.gov/Legislators/Profile/...,An Act regarding the disability pension for El...,Section 1. (a) Notwithstanding any general or ...,https://malegislature.gov/Bills/192/H5330,https://malegislature.gov/Bills/192/H5330/Hous...,https://malegislature.gov/Bills/192/H5330.pdf,...,Boston,0,[],"[{'date': '9/29/2022', 'branch': 'House', 'act...",../downloaded_legislation/state/MA/192nd/2025/...,Section 1. (a) Notwithstanding any general or ...,An Act regarding the disability pension for El...,disability pension elaina mcalister,0,0.502576
2067,H.4111,https://malegislature.gov/Bills/192/H4111,An Act authorizing the city of Boston to estab...,"Miranda, Liz",https://malegislature.gov/Legislators/Profile/...,An Act authorizing the city of Boston to estab...,Section 1. Paragraph (i) of subdivision (4) of...,https://malegislature.gov/Bills/192/H4111,https://malegislature.gov/Bills/192/H4111/Hous...,https://malegislature.gov/Bills/192/H4111.pdf,...,Boston,16,"[{'name': 'Thomas M. Stanley', 'district': '9t...","[{'date': '9/16/2021', 'branch': 'House', 'act...",../downloaded_legislation/state/MA/192nd/2025/...,NaN,An Act authorizing the city of Boston to estab...,authorize city boston alternative superannuati...,-1,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34559,H.3759,https://malegislature.gov/Bills/190/H3759,An Act relative to expanding access to high qu...,"Collins, Nick",https://malegislature.gov/Legislators/Profile/...,An Act relative to expanding access to high qu...,SECTION 1. Section 10 of chapter 152 of the ac...,https://malegislature.gov/Bills/19

Import legislator data

In [3]:
legislatorsPath =  os.path.join('..', 'data', 'raw', 'extended_legislators.csv')
legislatorsDF = pd.read_csv(legislatorsPath, engine='python')
legislatorsDF

,url,First Name,Last Name,District,Party,committee_ids
0,https://malegislature.gov/Legislators/Profile/...,James,Arciero,2nd Middlesex,Democrat,[30]
1,https://malegislature.gov/Legislators/Profile/...,James,Arena-DeRosa,8th Middlesex,Democrat,"[1, 16, 23, 31]"
2,https://malegislature.gov/Legislators/Profile/...,Jennifer Balinsky,Armini,8th Essex,Democrat,"[12, 27, 30, 49]"
3,https://malegislature.gov/Legislators/Profile/...,Shirley,Arriaga,8th Hampden,Democrat,"[15, 21, 31, 32, 55]"
4,https://malegislature.gov/Legislators/Profile/...,Brian,Ashe,2nd Hampden,Democrat,"[5, 29, 45, 54]"
...,...,...,...,...,...,...
448,https://malegislature.gov/Legislators/Profile/...,Marie,Fleur,NaN,NaN,NaN
449,https://malegislature.gov/Legislators/Profile/...,Richard,Ross,NaN,NaN,NaN
450,https://malegislature.gov/Legislators/Profile/...,Robert,Nyman,NaN,NaN,NaN
451,https://malegislature.gov/Legislators/Profile/...,William,Lantigua,NaN,NaN,NaN


In [4]:
# Mapping each bill and legislator to a unique node id
legislator_ids = list(legislatorsDF['url'].unique())
legislator_id_map = {url: idx for idx, url in enumerate(legislator_ids)}

bill_ids = list(billsDF['bill_url'])
bill_id_map = {url: idx + len(legislator_ids) for idx, url in enumerate(bill_ids)}

In [5]:
print(legislator_id_map)
print(bill_id_map)

{'https://malegislature.gov/Legislators/Profile/J_A1': 0, 'https://malegislature.gov/Legislators/Profile/JCD1': 1, 'https://malegislature.gov/Legislators/Profile/JBA1': 2, 'https://malegislature.gov/Legislators/Profile/SBA1': 3, 'https://malegislature.gov/Legislators/Profile/BMA1': 4, 'https://malegislature.gov/Legislators/Profile/BJA1': 5, 'https://malegislature.gov/Legislators/Profile/MLB1': 6, 'https://malegislature.gov/Legislators/Profile/CPB2': 7, 'https://malegislature.gov/Legislators/Profile/J_B1': 8, 'https://malegislature.gov/Legislators/Profile/DRB1': 9, 'https://malegislature.gov/Legislators/Profile/D_B1': 10, 'https://malegislature.gov/Legislators/Profile/NMB1': 11, 'https://malegislature.gov/Legislators/Profile/NAG1': 12, 'https://malegislature.gov/Legislators/Profile/AFC1': 13, 'https://malegislature.gov/Legislators/Profile/DFC1': 14, 'https://malegislature.gov/Legislators/Profile/S_C1': 15, 'https://malegislature.gov/Legislators/Profile/MSC1': 16, 'https://malegislature.

In [6]:

# Custom graph implemntation, adding bill nodes

mygraph = graph.CustomGraph()
for bill_url, bill_id  in bill_id_map.items():
    bill = billsDF[billsDF['bill_url'] == bill_url]
    billNode = graph.BillNode(bill_id, bill['bill_number'], bill_url, bill['title'], bill['text'], bill['sponsor'], bill['sponsor_url'], bill['view_text_url'], bill['download_pdf_url'], bill['is_home_rule'], bill['municipality'], bill['num_cosponsors'], bill['cosponsors'], bill['history'], bill['path'], bill['topic'])
    mygraph.add_bill(billNode)


In [7]:
# Custom graph implementation, adding legislator nodes
for legislator_url, legislator_id  in legislator_id_map.items():
    legislator = legislatorsDF[legislatorsDF['url'] == legislator_url]
    legislatorNode = graph.LegislatorNode(legislator_id, legislator_url, legislator['First Name'], legislator['Last Name'], legislator['District'], legislator['Party'], legislator['committee_ids'] )
    mygraph.add_legislator(legislatorNode)

In [8]:
unique_topics = billsDF['topic'].unique()
print(unique_topics)

[ 0  3 -1  1  5  7]


Add topic nodes

In [11]:
topic_id_map = {topicID: idx + len(legislator_ids) + len(bill_ids) for idx, topicID in enumerate(unique_topics)}
print(topic_id_map)
for topic_id, idx in topic_id_map.items():
    topic_node = graph.TopicNode(idx, topic_id)
    mygraph.add_topic(topic_node)

{0: 545, 3: 546, -1: 547, 1: 548, 5: 549, 7: 550}


In [12]:
mygraph.edges = []
for bill_id, bill in mygraph.bill_nodes.items():
    sponsor = bill.sponsor
    sponsor_url = bill.sponsor_url.iloc[0]
    #print(sponsor_url)
    sponsor_id = None

    #adding bill <-> topic
    topic_idx = bill.topic.iloc[0]
    if topic_idx in topic_id_map.keys():
        topic_id = topic_id_map[topic_idx]
        mygraph.add_edge(bill_id, topic_id, 'topic')
    

    #adding bill <-> sponsor
    if sponsor_url in legislator_id_map.keys():
        sponsor_id = legislator_id_map[sponsor_url]
        mygraph.add_edge(bill_id, sponsor_id, 'sponsor')
    #adding bill <-> cosponsor
    cosponsors_raw = bill.cosponsors.iloc[0]
    cosponsors = []
    if isinstance(cosponsors_raw, str):
        cosponsors = ast.literal_eval(cosponsors_raw)
    else:
        cosponsors = cosponsors_raw  # already parsed
    if len(cosponsors) > 0:
        for cosponsor in cosponsors:
            # print(cosponsor)
            cosponsor_url = cosponsor['url'].split('/1')[0]
            #print(cosponsor_url)
            if cosponsor_url in legislator_id_map.keys():
                cosponsor_id = legislator_id_map[cosponsor_url]
                mygraph.add_edge(bill_id, cosponsor_id, 'cosponsor')
    

In [13]:
print("Total edges:", len(mygraph.edges))
print("Sample edges:")
for edge in mygraph.edges[:5]:
    print(edge)

print(mygraph.edges[4])


Total edges: 341
Sample edges:
(453, 545, 'topic')
(453, 95, 'sponsor')
(454, 546, 'topic')
(454, 61, 'sponsor')
(455, 547, 'topic')
(455, 547, 'topic')


In [14]:
def get_subgraph(mygraph, root_id, depth=2):
    from collections import deque

    visited = set()
    sub_nodes = set()
    sub_edges = []

    queue = deque([(root_id, 0)])

    while queue:
        node_id, d = queue.popleft()
        if node_id not in visited:
            
            if d > depth:
                break
            visited.add(node_id)
            sub_nodes.add(node_id)

            for src, dst, rel in mygraph.edges:
                #print(src, dst, rel)
                if src == node_id or dst == node_id:
                    neighbor = dst if src == node_id else src
                    if neighbor not in visited and d + 1 <= depth:
                        sub_edges.append((src, dst, rel))
                        #print("Appending", neighbor)
                        queue.append((neighbor, d + 1))
        #else:
            #print("Node already visited")
    return sub_nodes, sub_edges


In [15]:
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network

def visualize_with_networkx(mygraph, root_id, depth=1):
    G = nx.DiGraph()

    sub_nodes, sub_edges = get_subgraph(mygraph, root_id, depth)
    #print(sub_nodes, sub_edges)

    G.add_node(str(root_id), label="Bill " + str(root_id), type='bill')

    # Add nodes
    print("Adding nodes")
    for node_id in sub_nodes:
        if node_id in mygraph.bill_nodes:
            bill_name = str(mygraph.bill_nodes[node_id].title.to_string())
            G.add_node(str(node_id), label="Bill " +  str(node_id), title = bill_name ,type='bill')
        elif node_id in mygraph.legislator_nodes:
            #legislator_id_map[node_id]
            first = str(mygraph.legislator_nodes[node_id].first.to_string().split()[1])
            last = str(mygraph.legislator_nodes[node_id].last.to_string().split()[1])
            party = str(mygraph.legislator_nodes[node_id].party.to_string().split()[1])
            nameAndParty = first + " " + last + ", " + party
            G.add_node(str(node_id), label="Legislator " + str(node_id), title =  (nameAndParty) , type='legislator')
        else:
            G.add_node(str(node_id), label = "Topic", type = "Topic")
    # Add edges
    print("Adding edges")
    for src, dst, rel in sub_edges:
        G.add_edge(str(src), str(dst), label=rel)


    # filter only for the nodes with the required degree
    min_degree = 0
    nodes_to_keep = [node for node, degree in G.degree() if degree >= min_degree]
    pruned_subgraph = G.subgraph(nodes_to_keep)

    # Draw
    print("Drawing")
    net = Network(height="800px", width="100%", notebook=True, bgcolor="#222222", font_color="white", cdn_resources="remote"  )
    net.force_atlas_2based(gravity=-200, central_gravity=0.01, spring_length=100)
    #net.toggle_physics(False)
    net.from_nx(pruned_subgraph)

    net.repulsion(node_distance=400, central_gravity=0.1)

    # Save and view
    # need to install jinja2 and pyvis
    net.show("graph.html")


sample_bill_id = bill_id_map['https://malegislature.gov/Bills/192/H5097']
print(sample_bill_id)
visualize_with_networkx(mygraph, sample_bill_id, 10)


453
Adding nodes
Adding edges
Drawing
graph.html
